# GPUHub 运行手册

建议选择 GPUHub 的 PyTorch / Miniconda 镜像，并使用 RTX 4080(S) / 32GB 实例。

Notebook 按顺序执行即可：
1. 克隆或更新仓库
2. 安装依赖
3. 检查 CUDA
4. 启动 Web 服务
5. 用手机访问实例服务地址


In [ ]:
REPO_URL = 'https://github.com/2192600891/driver-dms-v1.git'
REPO_DIR = '/root/driver-dms-v1'
PORT = 6006
DEVICE = '0'


In [ ]:
!if [ -d "$REPO_DIR/.git" ]; then \
  cd "$REPO_DIR" && git pull --ff-only; \
else \
  git clone "$REPO_URL" "$REPO_DIR"; \
fi

In [ ]:
%cd /root/driver-dms-v1
!pwd
!ls

## 安装依赖

如果你选择的是 GPUHub 官方 PyTorch 镜像，通常不需要再安装 PyTorch，只安装项目额外依赖即可。

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements-gpuhub.txt

如果镜像里没有 CUDA 版 PyTorch，再执行下面这个单元。已经有 CUDA 版时跳过。

请根据 GPUHub 镜像里的 CUDA 版本选择对应安装命令。

In [ ]:
# 可选：仅当 torch.cuda.is_available() 为 False 且你确认当前镜像缺少 CUDA 版 PyTorch 时再运行
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!python - <<'PY'
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device count:', torch.cuda.device_count())
    print('device 0:', torch.cuda.get_device_name(0))
PY

In [ ]:
!python - <<'PY'
import mydetect
print(mydetect.runtime_info())
PY

## 启动服务

启动后，Web 页面根地址为 `/`，健康检查为 `/api/health`，手机 H5 页面也在根地址。

In [ ]:
!pkill -f "uvicorn server_app:app" || true
!nohup env PORT=$PORT DMS_DEVICE=$DEVICE python -m uvicorn server_app:app --host 0.0.0.0 --port $PORT > gpuhub_server.log 2>&1 &
!sleep 5
!tail -n 30 gpuhub_server.log

In [ ]:
!curl -s http://127.0.0.1:6006/api/health

## 停止服务


In [ ]:
!pkill -f "uvicorn server_app:app" || true